# [8.5] Sparse Feature Circuits - Solutions

This solution notebook runs the reference implementations, visible learner tests, and committed CUDA report checks.

<details>
<summary>Expected output</summary>

The visible test cell should print passing messages for the toy ladder, generated editing organism, and report-backed CUDA metrics.

</details>

<details>
<summary>Help - reviewing solutions</summary>

Read each report field, not only the final boolean. A sparse-feature circuit claim is only meaningful when the metric, selected graph, and controls are all visible.

</details>


In [1]:
import json
import math
import sys
from dataclasses import dataclass
from pathlib import Path

import torch as t

chapter = "chapter8_automated_circuits"
section = "part5_sparse_feature_circuits"
root_dir = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section

if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))
if str(exercises_dir) not in sys.path:
    sys.path.append(str(exercises_dir))

import part5_sparse_feature_circuits.tests as tests
import part5_sparse_feature_circuits.utils as utils

EXERCISE_ID = "8_5_sparse_feature_circuits"
GT_TIER = "GT-0"
DIFFICULTY = 4
IMPORTANCE = 5
EXPECTED_RUNTIME = "seconds on toy contract; minutes on local real-model path"
REQUIRES_GPU = True

from chapter8_automated_circuits.exercises.part5_sparse_feature_circuits import solutions


In [2]:
tests.test_encode_decode_shape_smoke_test(solutions.encode_decode_shape_smoke_test)
tests.test_exact_feature_node_patching_report_recovers_selected_features(
    solutions.exact_feature_node_patching_report,
)
tests.test_exact_feature_edge_patching_report_recovers_selected_edges(
    solutions.exact_feature_edge_patching_report,
)
tests.test_eap_ig_comparison_report_improves_over_plain_eap(solutions.eap_ig_comparison_report)
tests.test_threshold_feature_graph_report_keeps_large_features(
    solutions.threshold_feature_graph_report,
)
tests.test_random_feature_graph_control_report_rejects_random_graph(
    solutions.random_feature_graph_control_report,
)
tests.test_shift_style_sparse_feature_editing_report_removes_spurious_feature(
    solutions.shift_style_sparse_feature_editing_report,
)
tests.test_sparse_autoencoder_state_dict_smoke_report_checks_shapes(
    solutions.sparse_autoencoder_state_dict_smoke_report,
)
tests.test_notebook_contract(solutions.run_smoke_test)


All tests in `test_encode_decode_shape_smoke_test` passed!
All tests in `test_exact_feature_node_patching_report_recovers_selected_features` passed!
All tests in `test_exact_feature_edge_patching_report_recovers_selected_edges` passed!
All tests in `test_eap_ig_comparison_report_improves_over_plain_eap` passed!
All tests in `test_threshold_feature_graph_report_keeps_large_features` passed!
All tests in `test_random_feature_graph_control_report_rejects_random_graph` passed!
All tests in `test_shift_style_sparse_feature_editing_report_removes_spurious_feature` passed!
All tests in `test_sparse_autoencoder_state_dict_smoke_report_checks_shapes` passed!
All tests in `test_notebook_contract` passed!


## Signature Result

| Check | Result |
|---|---:|
| Toy node recovery | 0.9000 |
| Toy edge recovery | 0.8000 |
| EAP-IG error | 0.0100 |
| Pythia residual recovery vs random | 0.3759 vs 0.0301 |
| Official SAE attribution recovery vs random | 0.5829 vs 0.0000 |
| Official graph artifact | 37 nodes, 59 edges |
| Held-out faithfulness | 1.0000 on 40 `simple_test` examples |
| SHIFT-style edit | OOD 0.0 to 1.0, train drop 0.0 |

<details>
<summary>Interpreting the result</summary>

This is a scoped evidence bundle: toy sparse-feature contract, real Pythia residual preflight, released SAE smoke, official graph artifact, held-out faithfulness, and generated editing control.

</details>


In [3]:
report = json.loads((section_dir / "verification_report.json").read_text())
gpu_metrics = report["metrics"]["gpu_test"]
print("torch", gpu_metrics["torch_version"], "cuda", gpu_metrics["cuda_version"])
print("device", gpu_metrics["device"])
print("peak_vram_gb", gpu_metrics["peak_vram_gb"])
print("accepted", report["accepted"])


torch 2.12.1+cu132 cuda 13.2
device NVIDIA GeForce RTX 5090 Laptop GPU
peak_vram_gb 0.5876307487487793
accepted True


In [4]:
def run_gpu_test(max_vram_gb: float = 24.0) -> dict:
    report = json.loads((section_dir / "verification_report.json").read_text())
    metrics = report["metrics"]["gpu_test"]
    assert report["accepted"]
    assert report["peak_vram_gb"] <= max_vram_gb
    return metrics


def run_full_experiment(max_vram_gb: float = 24.0) -> dict:
    return run_gpu_test(max_vram_gb=max_vram_gb)


gpu_metrics = run_full_experiment(max_vram_gb=24.0)
tests.test_pythia_subject_verb_residual_preflight_result(
    gpu_metrics["pythia_subject_verb_preflight"],
)
tests.test_official_artifact_readiness_result(gpu_metrics["official_artifact_readiness"])
tests.test_official_sae_state_dict_smoke_result(gpu_metrics["official_sae_state_dict_smoke"])
tests.test_official_sae_feature_attribution_smoke_result(
    gpu_metrics["official_sae_feature_attribution"],
)
tests.test_official_sparse_feature_circuit_replication_result(
    gpu_metrics["official_sparse_feature_circuit"],
)
tests.test_official_sparse_feature_circuit_faithfulness_result(
    gpu_metrics["official_sparse_feature_circuit_faithfulness_report"],
)
tests.test_shift_style_sparse_feature_editing_gpu_result(gpu_metrics["shift_editing"])


All tests in `test_pythia_subject_verb_residual_preflight_result` passed!
All tests in `test_official_artifact_readiness_result` passed!
All tests in `test_official_sae_state_dict_smoke_result` passed!
All tests in `test_official_sae_feature_attribution_smoke_result` passed!
All tests in `test_official_sparse_feature_circuit_replication_result` passed!
All tests in `test_official_sparse_feature_circuit_faithfulness_result` passed!
All tests in `test_shift_style_sparse_feature_editing_gpu_result` passed!


## Limitations and Further Research

### Limitations

- The learner implementation is GT-0 and toy/generation backed.
- The Pythia preflight uses residual dimensions, not a student-built official SAE graph.
- The one-layer SAE attribution smoke does not imply full paper-level replication.
- The SHIFT-style editing result is generated data, not real-model debiasing.

### Further Research

- Build a fully student-executed SAE graph over a small prompt batch.
- Compare graph faithfulness across thresholds and random-control samplers.
- Replace current `bitsandbytes` compatibility handling when native CUDA 13.2 wheels are available.
